In [1]:
# ─── CELL 1: mount & navigate ───────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.chdir("/content/drive/MyDrive/Multi-Agent-RAG-System-for-Enterprise-Document-Intelligence")
sys.path.insert(0, '.')
print("Working dir:", os.getcwd())

Mounted at /content/drive
Working dir: /content/drive/MyDrive/Multi-Agent-RAG-System-for-Enterprise-Document-Intelligence


In [45]:
# ─── CELL 2: create FastAPI app ──────────────────────────────────────────
api_content = '''# src/api.py
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import time, os, sys
sys.path.insert(0, ".")

app = FastAPI(
    title="Multi-Agent RAG Enterprise API",
    description="""
A production-ready Multi-Agent RAG system built with LangGraph.

## Architecture
Three specialised agents collaborate to answer questions:
- **Retriever Agent** — semantic vector search over document corpus
- **Analyst Agent** — extracts key facts from retrieved documents
- **Synthesiser Agent** — generates final grounded answer

## Tool Interface
Follows the Model Context Protocol (MCP) tool-calling pattern.
    """,
    version="1.0.0"
)

class QuestionRequest(BaseModel):
    question: str

class AgentPipelineResponse(BaseModel):
    question:      str
    final_answer:  str
    analyst_facts: str
    success:       bool
    time_sec:      float
    pipeline:      str = "retriever → analyst → synthesiser"

class HealthResponse(BaseModel):
    status:     str
    model:      str
    index_size: int
    agents:     list

from langchain_groq import ChatGroq
from src.graph import run_pipeline
from tools.retrieval_tool import index

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("Set GROQ_API_KEY environment variable before starting")

_llm = ChatGroq(
    api_key=GROQ_API_KEY,
    model="llama-3.1-8b-instant",
    temperature=0.1,
    max_tokens=512
)

@app.get("/")
def root():
    return {"name": "Multi-Agent RAG Enterprise API", "version": "1.0.0"}

@app.get("/health", response_model=HealthResponse)
def health_check():
    return HealthResponse(
        status="healthy", model="llama-3.1-8b-instant",
        index_size=index.ntotal,
        agents=["retriever", "analyst", "synthesiser"]
    )

@app.post("/ask", response_model=AgentPipelineResponse)
def ask(request: QuestionRequest):
    if not request.question.strip():
        raise HTTPException(status_code=400, detail="Question cannot be empty")
    if len(request.question) > 500:
        raise HTTPException(status_code=400, detail="Question too long")
    t0      = time.perf_counter()
    result  = run_pipeline(request.question, _llm)
    elapsed = round(time.perf_counter() - t0, 3)
    success = "cannot be determined" not in result["final_answer"].lower()
    return AgentPipelineResponse(
        question=request.question,
        final_answer=result["final_answer"],
        analyst_facts=result["analyst_facts"],
        success=success,
        time_sec=elapsed
    )
'''

with open("src/api.py", "w") as f:
    f.write(api_content)
print("api.py created.")

api.py created.


In [3]:
# ─── CELL 3: create API test script ─────────────────────────────────────
test_content = '''# src/test_api.py
"""
Tests the FastAPI endpoints.
Run AFTER starting server: uvicorn src.api:app --reload
"""
import requests, json

BASE_URL = "http://localhost:8000"

def test_root():
    print("=== ROOT ===")
    r = requests.get(f"{BASE_URL}/")
    print(json.dumps(r.json(), indent=2))

def test_health():
    print("\\n=== HEALTH ===")
    r = requests.get(f"{BASE_URL}/health")
    print(json.dumps(r.json(), indent=2))

def test_ask(question: str):
    print(f"\\n=== ASK ===")
    print(f"Q: {question}")
    r = requests.post(
        f"{BASE_URL}/ask",
        json={"question": question}
    )
    data = r.json()
    print(f"Status:  {r.status_code}")
    print(f"Answer:  {data.get(\'final_answer\', \'\')[:200]}")
    print(f"Success: {data.get(\'success\')}")
    print(f"Time:    {data.get(\'time_sec\')}s")

if __name__ == "__main__":
    test_root()
    test_health()
    test_ask("What is the Reserve Bank of Australia?")
    test_ask("What is risk-based authentication?")
'''

with open("src/test_api.py", "w") as f:
    f.write(test_content)
print("test_api.py created.")

test_api.py created.


In [4]:
# ─── CELL 4: create Dockerfile ───────────────────────────────────────────
dockerfile = '''FROM python:3.11-slim

WORKDIR /app

# system dependencies
RUN apt-get update && apt-get install -y \\
    build-essential \\
    && rm -rf /var/lib/apt/lists/*

# Python dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# copy source
COPY src/     ./src/
COPY agents/  ./agents/
COPY tools/   ./tools/
COPY data/    ./data/

# expose port
EXPOSE 8000

# start API
CMD ["uvicorn", "src.api:app", "--host", "0.0.0.0", "--port", "8000"]
'''

with open("Dockerfile", "w") as f:
    f.write(dockerfile)
print("Dockerfile created.")

Dockerfile created.


In [5]:
# ─── CELL 5: create docker-compose.yml ──────────────────────────────────
compose = '''version: "3.8"

services:
  multi-agent-api:
    build: .
    ports:
      - "8000:8000"
    environment:
      - GROQ_API_KEY=${GROQ_API_KEY}
    volumes:
      - ./data:/app/data
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3
'''

with open("docker-compose.yml", "w") as f:
    f.write(compose)
print("docker-compose.yml created.")

docker-compose.yml created.


In [6]:
# ─── CELL 6: create requirements.txt ────────────────────────────────────
requirements = """langchain>=1.0.0
langchain-community>=0.3.0
langchain-core>=0.3.0
langchain-groq>=0.2.0
langgraph>=0.2.0
groq>=0.9.0
fastapi>=0.110.0
uvicorn>=0.29.0
datasets>=2.18.0
pandas>=2.0.0
numpy>=1.24.0
sentence-transformers>=2.7.0
faiss-cpu>=1.7.4
python-dotenv>=1.0.0
matplotlib>=3.7.0
requests>=2.31.0
pydantic>=2.0.0
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)
print("requirements.txt created.")

requirements.txt created.


In [13]:
!pip install langchain langchain-community langchain-core \
             langchain-groq langgraph groq \
             faiss-cpu sentence-transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 62.7 MB/s eta 0:00:00


In [42]:
import subprocess, time, requests, os
from google.colab import userdata

GROQ_KEY = userdata.get("GROQ_API")  # changed from GROQ_TOKEN
os.system("pkill -f 'uvicorn src.api'")
time.sleep(2)

env = os.environ.copy()
env["GROQ_API_KEY"] = GROQ_KEY

server = subprocess.Popen(
    ["python", "-m", "uvicorn", "src.api:app",
     "--host", "0.0.0.0", "--port", "8000"],
    stderr=open("/content/server.log", "w"),
    stdout=open("/content/server.log", "a"),
    cwd="/content/drive/MyDrive/Multi-Agent-RAG-System-for-Enterprise-Document-Intelligence",
    env=env
)

print("Starting — waiting 35s...")
time.sleep(35)

if server.poll() is not None:
    print("CRASHED:", open("/content/server.log").read()[-500:])
else:
    print("Health:", requests.get("http://localhost:8000/health").json())
    print("Server ready.")

Starting — waiting 35s...
Health: {'status': 'healthy', 'model': 'llama-3.1-8b-instant', 'index_size': 4089, 'agents': ['retriever', 'analyst', 'synthesiser']}
Server ready.


In [43]:
import requests

questions = [
    "What is the Reserve Bank of Australia?",
    "What is risk-based authentication?"
]

for q in questions:
    print(f"\nQ: {q}")
    r = requests.post(
        "http://localhost:8000/ask",
        json={"question": q},
        timeout=120
    )
    print(f"Status:  {r.status_code}")
    if r.status_code == 200:
        data = r.json()
        print(f"Answer:  {data['final_answer'][:200]}")
        print(f"Success: {data['success']}")
        print(f"Time:    {data['time_sec']}s")
    else:
        print(f"Error: {r.text[:200]}")
    print("---")


Q: What is the Reserve Bank of Australia?
Status:  200
Answer:  The Reserve Bank of Australia (RBA) is Australia's central bank and banknote issuing authority, established on 14 January 1960. It is responsible for managing the country's monetary policy and has ass
Success: True
Time:    0.703s
---

Q: What is risk-based authentication?
Status:  200
Answer:  Risk-based authentication (RBA) is a method of applying varying levels of stringency to authentication processes based on the likelihood that access to a given system could result in its being comprom
Success: True
Time:    0.687s
---


In [44]:
server.terminate()
print("Server stopped.")

Server stopped.


In [ ]:
!git config --global user.email "chaitanyamhetre97@email.com"
!git config --global user.name "chaitanyamhetre"

!git add .
!git commit -m "Day 4 — FastAPI working, all endpoints tested"
!git push
print("Pushed.")